In [0]:

BEGIN
  DECLARE v_month_start     DATE;
  DECLARE v_month_end       DATE;
  DECLARE v_prior_month_end DATE;
  DECLARE v_end             DATE;

  SET v_month_start = (
    SELECT add_months(MAX(report_month), 1)
    FROM data_exploration_hub.marketing_hub.mkt_dashboard_demographics_monthly
  );
  SET v_end = (
    SELECT LEAST(
      (SELECT to_date(CAST(MAX(snapshot_date_id) AS STRING), 'yyyyMMdd')
         FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot),
      (SELECT to_date(CAST(MAX(SnapshotDateId) AS STRING), 'yyyyMMdd')
         FROM cpx_dataanalytics_gold.customer360.c360_f_rfm_model),
      date_sub(current_date(), 1)
    )
  );

  WHILE last_day(v_month_start) <= v_end DO
    SET v_month_end = last_day(v_month_start);
    SET v_prior_month_end = date_sub(v_month_start, 1);

    INSERT INTO data_exploration_hub.marketing_hub.mkt_dashboard_demographics_monthly
    WITH signups_pop AS (
        SELECT DISTINCT cc_user_profile_id
        FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
        WHERE event_type = 'subscription_created'
          AND plan_id IN ('monthly1','annual1','gift-plan')
          AND reporting_datetime BETWEEN v_month_start AND v_month_end
    ),
    bal_monthly_active AS (
        SELECT DISTINCT cc_user_profile_id
        FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
        WHERE event_type = 'payment_succeeded' AND plan_id = 'monthly1'
          AND v_month_end BETWEEN CAST(current_term_start_datetime AS DATE) AND CAST(current_term_end_datetime AS DATE)
    ),
    bal_annual_active AS (
        SELECT DISTINCT cc_user_profile_id
        FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
        WHERE event_type IN ('subscription_created','subscription_renewed','subscription_resumed','subscription_started')
          AND plan_id = 'annual1'
          AND v_month_end BETWEEN CAST(current_term_start_datetime AS DATE) AND CAST(current_term_end_datetime AS DATE)
    ),
    bal_gift_active AS (
        SELECT DISTINCT cc_user_profile_id
        FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
        WHERE event_type IN ('subscription_created','subscription_renewed','subscription_resumed','subscription_started')
          AND plan_id = 'gift-plan'
          AND v_month_end BETWEEN CAST(current_term_start_datetime AS DATE) AND CAST(current_term_end_datetime AS DATE)
    ),
    bal_last_payment AS (
        SELECT cc_user_profile_id, event_type, reporting_datetime, current_term_start_datetime,
               ROW_NUMBER() OVER (PARTITION BY cc_user_profile_id ORDER BY reporting_datetime DESC) AS rn
        FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
        WHERE event_type IN ('payment_failed','payment_succeeded') AND plan_id = 'monthly1'
          AND reporting_datetime <= v_month_end
    ),
    bal_last_succeeded AS (
        SELECT cc_user_profile_id, MAX(current_term_start_datetime) AS last_succeeded_term_start
        FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
        WHERE event_type = 'payment_succeeded' AND plan_id = 'monthly1'
        GROUP BY cc_user_profile_id
    ),
    bal_monthly_dunning AS (
        SELECT DISTINCT lp.cc_user_profile_id
        FROM bal_last_payment lp
        LEFT JOIN bal_last_succeeded ls ON lp.cc_user_profile_id = ls.cc_user_profile_id
        WHERE lp.rn = 1 AND lp.event_type = 'payment_failed'
          AND lp.reporting_datetime >= DATE_SUB(v_month_end, 15)
          AND lp.reporting_datetime <= v_month_end
          AND CAST(lp.reporting_datetime AS DATE) = CAST(lp.current_term_start_datetime AS DATE)
          AND (ls.last_succeeded_term_start IS NULL OR ls.last_succeeded_term_start < lp.current_term_start_datetime)
    ),
    a_last_payment AS (
        SELECT cc_user_profile_id, event_type, reporting_datetime, current_term_start_datetime,
               ROW_NUMBER() OVER (PARTITION BY cc_user_profile_id ORDER BY reporting_datetime DESC) AS rn
        FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
        WHERE event_type IN ('payment_failed','payment_succeeded') AND plan_id = 'annual1'
          AND reporting_datetime <= v_month_end
    ),
    a_last_succeeded AS (
        SELECT cc_user_profile_id, MAX(current_term_start_datetime) AS last_succeeded_term_start
        FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
        WHERE event_type = 'payment_succeeded' AND plan_id = 'annual1'
        GROUP BY cc_user_profile_id
    ),
    a_eom_dunning AS (
        SELECT DISTINCT lp.cc_user_profile_id
        FROM a_last_payment lp
        LEFT JOIN a_last_succeeded ls ON lp.cc_user_profile_id = ls.cc_user_profile_id
        WHERE lp.rn = 1 AND lp.event_type = 'payment_failed'
          AND lp.reporting_datetime >= DATE_SUB(v_month_end, 14)
          AND lp.reporting_datetime <= v_month_end
          AND (ls.last_succeeded_term_start IS NULL OR ls.last_succeeded_term_start < lp.current_term_start_datetime)
          AND CAST(lp.reporting_datetime AS DATE) = CAST(lp.current_term_start_datetime AS DATE)
    ),
    ptd_balance_pop AS (
        SELECT cc_user_profile_id FROM bal_monthly_active
        UNION SELECT cc_user_profile_id FROM bal_annual_active
        UNION SELECT cc_user_profile_id FROM bal_gift_active
        UNION SELECT cc_user_profile_id FROM bal_monthly_dunning
        UNION SELECT cc_user_profile_id FROM a_eom_dunning
    ),
    active_balance_cde AS (
        SELECT DISTINCT s.cde_id
        FROM ptd_balance_pop p
        JOIN cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot s
          ON s.cc_user_profile_id = p.cc_user_profile_id
         AND s.snapshot_date_id = date_format(v_month_end, 'yyyyMMdd')
    ),
    scene_active_cde AS (
        SELECT DISTINCT CDE_ID
        FROM cpx_dataanalytics_gold.customer360.c360_f_transaction
        WHERE SCENEFLAG = 'Yes'
          AND VisitDateId BETWEEN CAST(date_format(add_months(v_month_end, -12), 'yyyyMMdd') AS INT)
                              AND CAST(date_format(v_month_end, 'yyyyMMdd') AS INT)
    ),
    scene_balance_pop AS (
        SELECT sc.CDE_ID AS cde_id
        FROM scene_active_cde sc
        LEFT ANTI JOIN active_balance_cde ab ON CAST(sc.CDE_ID AS STRING) = ab.cde_id
    ),

    -- RFM: cc_user_profile_id -> cde_id bridge (EOM snapshot, prior-EOM fallback for annual lapsed)
    cde_lookup_eom AS (
        SELECT DISTINCT cc_user_profile_id, cde_id
        FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
        WHERE snapshot_date_id = date_format(v_month_end, 'yyyyMMdd')
    ),
    cde_lookup_prior_eom AS (
        SELECT DISTINCT cc_user_profile_id, cde_id
        FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
        WHERE snapshot_date_id = date_format(v_prior_month_end, 'yyyyMMdd')
    ),
    cde_lookup AS (
        SELECT cc_user_profile_id, cde_id FROM cde_lookup_eom
        UNION ALL
        SELECT p.cc_user_profile_id, p.cde_id
        FROM cde_lookup_prior_eom p
        LEFT ANTI JOIN cde_lookup_eom t ON t.cc_user_profile_id = p.cc_user_profile_id
    ),
    rfm_snapshot AS (
        SELECT CDE_ID, Segment
        FROM cpx_dataanalytics_gold.customer360.c360_f_rfm_model
        WHERE SnapshotDateId = date_format(v_month_end, 'yyyyMMdd')
    ),
    member_category_map AS (
        SELECT cl.cc_user_profile_id, COALESCE(rs.Segment, 'Dormant') AS rfm_category
        FROM cde_lookup cl
        LEFT JOIN rfm_snapshot rs ON CAST(cl.cde_id AS INT) = rs.CDE_ID
    ),

    demo AS (
        SELECT 'program_monthly_signups' AS metric_type, c.gender_text, c.age_cohort_id, c.province,
               COALESCE(mcm.rfm_category, 'Dormant') AS rfm_category
        FROM signups_pop p
        LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_customer c ON c.cc_user_profile_id = p.cc_user_profile_id
        LEFT JOIN member_category_map mcm ON mcm.cc_user_profile_id = p.cc_user_profile_id

        UNION ALL

        SELECT 'ptd_balance' AS metric_type, c.gender_text, c.age_cohort_id, c.province,
               COALESCE(mcm.rfm_category, 'Dormant') AS rfm_category
        FROM ptd_balance_pop p
        LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_customer c ON c.cc_user_profile_id = p.cc_user_profile_id
        LEFT JOIN member_category_map mcm ON mcm.cc_user_profile_id = p.cc_user_profile_id

        UNION ALL

        SELECT 'scene_balance' AS metric_type, c.gender_text, c.age_cohort_id, c.province,
               COALESCE(rs.Segment, 'Dormant') AS rfm_category
        FROM scene_balance_pop p
        LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_customer c ON CAST(c.cde_id AS STRING) = p.cde_id
        LEFT JOIN rfm_snapshot rs ON CAST(p.cde_id AS INT) = rs.CDE_ID
    )
    SELECT
        v_month_start AS report_month,
        metric_type,
        COUNT(IF(gender_text = 'Male', 1, NULL))   AS gender_male,
        COUNT(IF(gender_text = 'Female', 1, NULL)) AS gender_female,
        COUNT(IF(gender_text NOT IN ('Male','Female') OR gender_text IS NULL, 1, NULL)) AS gender_other,
        COUNT(IF(age_cohort_id = '1', 1, NULL)) AS age_14_18,
        COUNT(IF(age_cohort_id = '2', 1, NULL)) AS age_19_24,
        COUNT(IF(age_cohort_id = '3', 1, NULL)) AS age_25_34,
        COUNT(IF(age_cohort_id = '4', 1, NULL)) AS age_35_49,
        COUNT(IF(age_cohort_id = '5', 1, NULL)) AS age_50_64,
        COUNT(IF(age_cohort_id = '6', 1, NULL)) AS age_65_plus,
        COUNT(IF(age_cohort_id = '0' OR age_cohort_id IS NULL OR age_cohort_id NOT IN ('1','2','3','4','5','6'), 1, NULL)) AS age_unknown,
        COUNT(IF(UPPER(TRIM(province)) = 'ON', 1, NULL)) AS prov_on,
        COUNT(IF(UPPER(TRIM(province)) = 'BC', 1, NULL)) AS prov_bc,
        COUNT(IF(UPPER(TRIM(province)) = 'AB', 1, NULL)) AS prov_ab,
        COUNT(IF(UPPER(TRIM(province)) = 'QC', 1, NULL)) AS prov_qc,
        COUNT(IF(UPPER(TRIM(province)) = 'MB', 1, NULL)) AS prov_mb,
        COUNT(IF(UPPER(TRIM(province)) = 'SK', 1, NULL)) AS prov_sk,
        COUNT(IF(UPPER(TRIM(province)) IN ('NB','NS','PE','PEI','NL'), 1, NULL)) AS prov_atl,
        COUNT(IF(UPPER(TRIM(province)) NOT IN ('ON','BC','AB','QC','MB','SK','NB','NS','PE','PEI','NL')
                 OR province IS NULL, 1, NULL)) AS prov_other,
        COUNT(IF(rfm_category = 'Champions', 1, NULL)) AS rfm_champions,
        COUNT(IF(rfm_category = 'Potential Loyalist', 1, NULL)) AS rfm_potential_loyalist,
        COUNT(IF(rfm_category = 'Value Conscious Loyals', 1, NULL)) AS rfm_value_conscious_loyals,
        COUNT(IF(rfm_category IN ('Champions','Potential Loyalist','Value Conscious Loyals'), 1, NULL)) AS rfm_top3,
        COUNT(IF(rfm_category = 'Promising', 1, NULL)) AS rfm_promising,
        COUNT(IF(rfm_category = 'At Risk', 1, NULL)) AS rfm_at_risk,
        COUNT(IF(rfm_category = 'New', 1, NULL)) AS rfm_new,
        COUNT(IF(rfm_category = 'Fleeting', 1, NULL)) AS rfm_fleeting,
        COUNT(IF(rfm_category = 'High Value Lapsed', 1, NULL)) AS rfm_high_value_lapsed,
        COUNT(IF(rfm_category = 'Dormant' OR rfm_category IS NULL, 1, NULL)) AS rfm_dormant
    FROM demo
    GROUP BY metric_type
    ORDER BY CASE metric_type
        WHEN 'program_monthly_signups' THEN 1
        WHEN 'ptd_balance' THEN 2
        WHEN 'scene_balance' THEN 3
    END;

    SET v_month_start = add_months(v_month_start, 1);
END WHILE;
END;